# Loss Functions

Companion notebook for the [Loss Functions lesson](https://ml-viz-ruby.vercel.app/courses/optimization-ml/04-loss-functions).

We plot and compare the common losses, show empirically that **MSE predicts the mean while MAE
predicts the median** (and why that makes MAE robust to outliers), and verify the **MLE ↔ loss**
link. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## Intuition — the loss encodes your assumptions

The loss function is the single thing training minimizes, and *choosing* it is a modeling
decision, not a detail. Each loss quietly assumes a noise model: **MSE** assumes Gaussian noise
(and so chases the **mean**), **MAE** assumes heavier-tailed Laplace noise (and chases the
**median**, making it robust to outliers), and **cross-entropy** is the negative log-likelihood
of a Bernoulli label (so it's the natural classification loss). The loss also shapes the
gradients — how hard the model is pushed when it's wrong. We plot them, prove the mean/median
claim, and connect each back to maximum likelihood.

## 1 — Regression losses vs. the error

MSE grows quadratically (outlier-sensitive); MAE grows linearly (robust); Huber is quadratic near 0
and linear beyond δ — a smooth compromise.

In [ ]:
def mse(e): return e**2
def mae(e): return np.abs(e)
def huber(e, d=1.0):
    a = np.abs(e)
    return np.where(a <= d, 0.5*e**2, d*a - 0.5*d**2)

e = np.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(e, mse(e), label='MSE (L2)', color='#fb7185')
ax.plot(e, mae(e), label='MAE (L1)', color='#2dd4bf')
ax.plot(e, huber(e), label='Huber (δ=1)', color='#818cf8')
ax.set_xlabel('error  (ŷ − y)'); ax.set_ylabel('loss')
ax.set_title('Regression losses: MSE punishes large errors hardest'); ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**What to notice:** **MSE** curves upward quadratically — a 2× larger error is punished 4×,
so outliers dominate. **MAE** rises linearly (every error weighted equally). **Huber** is MSE near
zero (smooth gradients where it matters) and MAE far out (robust to big errors) — the practical
compromise for regression with outliers.

## 2 — MSE → mean, MAE → median (robustness)

The constant that minimizes total MSE is the mean; the constant that minimizes total MAE is the
median. With an outlier added, the mean lurches but the median barely moves — that's MAE's
robustness, made concrete.

In [ ]:
data = np.array([2.0, 3.0, 4.0, 5.0, 6.0])
outlier = np.append(data, 100.0)
for name, d in [('clean', data), ('with outlier', outlier)]:
    print(f'{name:13s}: MSE-minimizer (mean)={d.mean():6.2f}   MAE-minimizer (median)={np.median(d):.2f}')
print('\nThe outlier drags the mean to ~20 but the median stays ~4.5 -> MAE is robust.')

**What to notice:** adding a single outlier (100) drags the **mean** from ~4 to ~20 but leaves
the **median** near 4.5. Since MSE is minimized by the mean and MAE by the median, this is exactly
why MAE-trained models are **robust**: the loss you pick determines which summary statistic you
end up predicting.

## 3 — Cross-entropy punishes confident wrong predictions

For the true class with predicted probability p, the loss is −log p. As p→0 (confidently wrong) the
loss → ∞, producing strong gradients exactly when the model is badly mistaken.

In [ ]:
def binary_cross_entropy(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y*np.log(p) + (1-y)*np.log(1-p))

p = np.linspace(0.001, 0.999, 200)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(p, binary_cross_entropy(p, 1), color='#818cf8')
ax.set_xlabel('predicted probability of the TRUE class'); ax.set_ylabel('cross-entropy loss')
ax.set_title('Cross-entropy: tiny when confident-correct, huge when confident-wrong')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print('p=0.9 -> loss', round(float(binary_cross_entropy(0.9,1)),3))
print('p=0.1 -> loss', round(float(binary_cross_entropy(0.1,1)),3), '(confidently wrong = big loss)')

**What to notice:** cross-entropy is near 0 when the model is confident and correct
(`p=0.9 → 0.11`) but blows up when confident and wrong (`p=0.1 → 2.3`, heading to ∞ as `p→0`). That
steep tail produces **large gradients exactly when the model is badly mistaken** — a self-correcting
pressure the squared error on probabilities wouldn't give.

## 4. The library way — validate against `sklearn` and an optimizer

Frameworks ship these losses (`sklearn.metrics`, `torch.nn`). The cell checks our cross-entropy
against `sklearn.metrics.log_loss`, and uses `scipy.optimize` to confirm the headline claim — the
constant minimizing MSE is the **mean**, and the one minimizing MAE is the **median**.

In [ ]:
from sklearn.metrics import log_loss
from scipy.optimize import minimize_scalar

# our binary cross-entropy matches sklearn's log_loss
pv = np.array([0.9, 0.1, 0.8, 0.3]); yv = np.array([1, 0, 1, 0])
ours = np.mean([binary_cross_entropy(pi, yi) for pi, yi in zip(pv, yv)])
assert np.isclose(ours, log_loss(yv, pv)), "our BCE must match sklearn.log_loss"
print(f'BCE: ours = {ours:.4f}, sklearn.log_loss = {log_loss(yv, pv):.4f} ✓')

# MSE-minimizer = mean, MAE-minimizer = median (found by an optimizer, no formula)
data = np.array([2., 3., 4., 5., 100.])
c_mse = minimize_scalar(lambda t: np.mean((data - t)**2)).x
c_mae = minimize_scalar(lambda t: np.mean(np.abs(data - t))).x
print(f'MSE-min c = {c_mse:.3f}  (mean = {data.mean():.3f})')
print(f'MAE-min c = {c_mae:.3f}  (median = {np.median(data):.3f})')
assert np.isclose(c_mse, data.mean(), atol=1e-3) and abs(c_mae - np.median(data)) < 0.5
print('MSE->mean, MAE->median confirmed by direct optimization ✓')

**What to notice:** our cross-entropy equals `sklearn.log_loss` exactly, and the optimizer
independently lands the MSE minimizer on the **mean** (22.8, dragged up by the outlier) and the
MAE minimizer on the **median** (4). The "which statistic does this loss predict?" claim isn't
folklore — it falls out of the optimization.

## 5. Gotchas & tradeoffs

- **MSE is outlier-sensitive** (quadratic tail); **MAE is robust** but **non-differentiable at 0**
  (subgradient there) and has constant gradient magnitude, which can slow convergence. **Huber**
  blends both, at the cost of a `δ` to tune.
- **Cross-entropy needs a `log(0)` guard** — clip predicted probabilities away from 0 and 1 (or use
  a fused logits loss).
- **Loss = noise model.** MSE ⇔ Gaussian, MAE ⇔ Laplace, CE ⇔ Bernoulli. Pick the loss that matches
  your error distribution and metric, not by habit.
- **Train-loss ≠ eval-metric.** You optimize a differentiable surrogate (cross-entropy) but often
  *care* about a non-differentiable metric (accuracy, F1).

In [ ]:
# MSE vs MAE gradient magnitude: MSE's grad shrinks near 0 (fine-tuning), MAE's is constant
for e in [2.0, 0.5, 0.05]:
    print(f'error={e:>4}:  MSE grad |2e| = {abs(2*e):.3f}   MAE grad = 1.000 (constant)')
print('\n-> MSE eases off near the optimum; MAE keeps pushing at full strength (can oscillate)')

**What to notice:** MSE's gradient `2e` shrinks as the error shrinks, so training naturally
settles; MAE's gradient stays `±1` right down to zero error, which is robust but can cause
jitter near the optimum. The loss shapes not just *what* you predict but *how* training
behaves.

## Key takeaways

- The loss is a **modeling choice** encoding a noise model: **MSE↔Gaussian↔mean**,
  **MAE↔Laplace↔median**, **cross-entropy↔Bernoulli**.
- **MSE** punishes outliers hardest; **MAE** is robust but non-smooth; **Huber** compromises.
- **Cross-entropy** explodes on confident-wrong predictions (big corrective gradients) and needs a
  `log(0)` guard.
- Validate with `sklearn.metrics` / `torch.nn`; remember the **training loss** is a surrogate for
  the **eval metric** you actually care about.

**Next:** [Hyperparameter Optimization](https://ml-viz-ruby.vercel.app/courses/optimization-ml/05-hyperparameter-optimization).

## ✏️ Your turn

**Exercise.** Implement `huber_loss(e, delta)` (quadratic for |e|≤δ, linear beyond) and
`cross_entropy(p, y)` (binary, numerically stable via clipping). These are two of the most-used
losses in all of ML.

In [ ]:
def huber_loss(e, delta=1.0):
    # TODO(you): 0.5*e^2 if |e|<=delta else delta*|e| - 0.5*delta^2  (works on numpy arrays)
    return ...

def cross_entropy(p, y):
    # TODO(you): binary cross-entropy -[y log p + (1-y) log(1-p)], clip p for stability
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(huber_loss(np.array([0.5]), 1.0)[0], 0.125)        # quadratic region
assert np.isclose(huber_loss(np.array([3.0]), 1.0)[0], 1.0*3 - 0.5)  # linear region
assert np.isclose(cross_entropy(0.9, 1), -np.log(0.9))
assert cross_entropy(0.1, 1) > cross_entropy(0.9, 1)                  # confident-wrong costs more
print('\u2713 Huber and cross-entropy are correct')

<details>
<summary>Solution</summary>

```python
def huber_loss(e, delta=1.0):
    a = np.abs(e)
    return np.where(a <= delta, 0.5*e**2, delta*a - 0.5*delta**2)

def cross_entropy(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y*np.log(p) + (1-y)*np.log(1-p))
```

Huber is MSE near zero (smooth gradients) and MAE far out (robust to outliers). Cross-entropy is the
negative log-likelihood of a Bernoulli label — which is why it's the natural classification loss.

</details>